In [0]:
%run ../99_utils/utils_config

In [0]:
%run ../99_utils/utils_logger

In [0]:
%run ../99_utils/utils_table_logger

PROJECT CONFIGURATION LOADED SUCCESSFULLY
PROJECT_NAME: brazil_legislative_analytics
PROJECT_VERSION: v1.0.0
PROJECT_ENVIRONMENT: dev
CATALOG_NAME: brazil_legislative_analytics
RUN_ID: 043cd2e0-88f0-41d1-bb2a-79de2b280657


In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 02 Job — Run Incremental Expenses CEAP
# MAGIC
# MAGIC Orchestrates the incremental execution flow for CEAP expenses in the Brazil Legislative Analytics Medallion project.
# MAGIC
# MAGIC ## Purpose
# MAGIC This notebook executes only the pipeline steps required to refresh CEAP expenses data.
# MAGIC
# MAGIC ## Execution Flow
# MAGIC - Validate project setup
# MAGIC - Optionally validate API connection
# MAGIC - Execute Bronze CEAP expenses ingestion
# MAGIC - Execute Silver CEAP expenses transformation
# MAGIC - Execute Silver suppliers transformation
# MAGIC - Execute Gold supplier dimension
# MAGIC - Execute Gold CEAP expenses fact
# MAGIC - Execute CEAP analytical mart
# MAGIC - Execute quality checks
# MAGIC
# MAGIC ## Current Development Note
# MAGIC Some notebooks may still be under development.
# MAGIC In this case, the corresponding execution blocks can remain disabled until the notebooks are available.
# MAGIC
# MAGIC ## API Validation Note
# MAGIC API validation is intentionally disabled by default because external API endpoints may be unstable or slow in Databricks Free Edition.
# MAGIC
# MAGIC ## Documentation Standard
# MAGIC - Python functions and variables are written in English.
# MAGIC - Table and field names follow Portuguese mnemonic standards.
# MAGIC - Comments and documentation are written in English.

# COMMAND ----------

# MAGIC %run ../99_utils/utils_config

# COMMAND ----------

# MAGIC %run ../99_utils/utils_logger

# COMMAND ----------

# MAGIC %run ../99_utils/utils_table_logger

# COMMAND ----------

from datetime import datetime
import uuid
import traceback

# COMMAND ----------

print("=" * 90)
print("BRAZIL LEGISLATIVE ANALYTICS MEDALLION")
print("02 - RUN INCREMENTAL EXPENSES CEAP")
print("=" * 90)
print(f"Execution Timestamp: {datetime.now()}")
print(f"Catalog: {CATALOG_NAME}")
print(f"Environment: {PROJECT_ENVIRONMENT}")
print("=" * 90)

# COMMAND ----------

# ============================================================
# JOB CONFIGURATION
# ============================================================

NOTEBOOK_NAME = "02_run_incremental_expenses_ceap"
LAYER_NAME = "jobs"
ENTITY_NAME = "incremental_expenses_ceap"

JOB_RUN_ID = str(uuid.uuid4())

# During development, keep this as False to avoid stopping
# the job when not-yet-built notebooks are missing.
FAIL_ON_STEP_ERROR = False

PIPELINE_LOG_TABLE = (
    f"{CATALOG_NAME}."
    f"{SCHEMA_AUDIT}."
    f"{AUD_TB_LOG_EXECUCAO_PIPELINE}"
)

PIPELINE_ERROR_TABLE = (
    f"{CATALOG_NAME}."
    f"{SCHEMA_AUDIT}."
    f"{AUD_TB_LOG_ERROS_PIPELINE}"
)

logger = get_logger(
    logger_name=NOTEBOOK_NAME,
    layer_name=LAYER_NAME,
)

# COMMAND ----------

# ============================================================
# INCREMENTAL STEP CONFIGURATION
# ============================================================
#
# Set enabled=True only when the referenced notebook exists and
# is ready to execute.
#
# API validation is disabled by default in this job because the
# external API may be slow or unstable.
#
# ============================================================

INCREMENTAL_STEPS = [
    {
        "step_name": "validate_project_setup",
        "notebook_path": "../00_setup/90_validate_project_setup",
        "layer_name": "setup",
        "entity_name": "project_setup",
        "enabled": True,
        "critical": True,
    },
    {
        "step_name": "validate_api_connection",
        "notebook_path": "../00_setup/92_validate_api_connection",
        "layer_name": "setup",
        "entity_name": "api_connection",
        "enabled": False,
        "critical": False,
    },

    # Bronze CEAP ingestion.
    {
        "step_name": "bronze_despesas_ceap",
        "notebook_path": "../01_bronze/07_bronze_despesas_ceap",
        "layer_name": "bronze",
        "entity_name": "despesas_ceap",
        "enabled": False,
        "critical": True,
    },

    # Silver CEAP transformation.
    {
        "step_name": "silver_despesas_ceap",
        "notebook_path": "../02_silver/09_silver_despesas_ceap",
        "layer_name": "silver",
        "entity_name": "despesas_ceap",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "silver_fornecedores",
        "notebook_path": "../02_silver/10_silver_fornecedores",
        "layer_name": "silver",
        "entity_name": "fornecedores",
        "enabled": False,
        "critical": True,
    },

    # Gold CEAP modeling.
    {
        "step_name": "gold_dm_fornecedor",
        "notebook_path": "../04_gold/11_dm_fornecedor",
        "layer_name": "gold",
        "entity_name": "dm_fornecedor",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "gold_ft_despesas_ceap",
        "notebook_path": "../04_gold/16_ft_despesas_ceap",
        "layer_name": "gold",
        "entity_name": "ft_despesas_ceap",
        "enabled": False,
        "critical": True,
    },

    # CEAP analytical mart.
    {
        "step_name": "mart_panorama_despesas_ceap",
        "notebook_path": "../05_marts/04_am_panorama_despesas_ceap",
        "layer_name": "marts",
        "entity_name": "panorama_despesas_ceap",
        "enabled": False,
        "critical": True,
    },

    # Quality checks.
    {
        "step_name": "quality_bronze_checks",
        "notebook_path": "../06_quality/01_quality_bronze_checks",
        "layer_name": "quality",
        "entity_name": "bronze_checks",
        "enabled": True,
        "critical": False,
    },
    {
        "step_name": "quality_silver_checks",
        "notebook_path": "../06_quality/02_quality_silver_checks",
        "layer_name": "quality",
        "entity_name": "silver_checks",
        "enabled": True,
        "critical": False,
    },
    {
        "step_name": "quality_gold_checks",
        "notebook_path": "../06_quality/03_quality_gold_checks",
        "layer_name": "quality",
        "entity_name": "gold_checks",
        "enabled": True,
        "critical": False,
    },
    {
        "step_name": "quality_traceability_checks",
        "notebook_path": "../06_quality/04_traceability_checks",
        "layer_name": "quality",
        "entity_name": "traceability_checks",
        "enabled": True,
        "critical": False,
    },
]

# COMMAND ----------

# ============================================================
# JOB HELPERS
# ============================================================

def run_notebook_step(
    step_config: dict,
) -> dict:
    """
    Executes a configured notebook step and returns execution metadata.
    """

    step_name = step_config["step_name"]
    notebook_path = step_config["notebook_path"]
    step_layer = step_config["layer_name"]
    step_entity = step_config["entity_name"]
    is_enabled = step_config["enabled"]
    is_critical = step_config["critical"]

    step_log_id = str(uuid.uuid4())
    started_at = datetime.now()

    if not is_enabled:

        finished_at = datetime.now()
        duration_seconds = (
            finished_at - started_at
        ).total_seconds()

        write_pipeline_log(
            log_id=step_log_id,
            execution_id=JOB_RUN_ID,
            notebook_name=NOTEBOOK_NAME,
            layer_name=step_layer,
            entity_name=step_entity,
            target_table="not_applicable",
            status=EXECUTION_STATUS_WARNING,
            message=f"Step skipped because it is disabled: {step_name}",
            started_at=started_at,
            finished_at=finished_at,
            duration_seconds=duration_seconds,
            records_read=None,
            records_written=None,
        )

        log_warning(
            pipeline_logger=logger,
            message=f"Step skipped: {step_name}",
        )

        return {
            "step_name": step_name,
            "status": EXECUTION_STATUS_WARNING,
            "message": "Step skipped because it is disabled.",
            "duration_seconds": duration_seconds,
            "critical": is_critical,
        }

    try:

        write_pipeline_log(
            log_id=step_log_id,
            execution_id=JOB_RUN_ID,
            notebook_name=NOTEBOOK_NAME,
            layer_name=step_layer,
            entity_name=step_entity,
            target_table="not_applicable",
            status=EXECUTION_STATUS_STARTED,
            message=f"Step execution started: {step_name}",
            started_at=started_at,
            finished_at=None,
            duration_seconds=None,
            records_read=None,
            records_written=None,
        )

        log_info(
            pipeline_logger=logger,
            message=f"Starting step: {step_name}",
        )

        dbutils.notebook.run(
            notebook_path,
            timeout_seconds=0,
        )

        finished_at = datetime.now()
        duration_seconds = (
            finished_at - started_at
        ).total_seconds()

        write_pipeline_log(
            log_id=str(uuid.uuid4()),
            execution_id=JOB_RUN_ID,
            notebook_name=NOTEBOOK_NAME,
            layer_name=step_layer,
            entity_name=step_entity,
            target_table="not_applicable",
            status=EXECUTION_STATUS_SUCCESS,
            message=f"Step completed successfully: {step_name}",
            started_at=started_at,
            finished_at=finished_at,
            duration_seconds=duration_seconds,
            records_read=None,
            records_written=None,
        )

        log_success(
            pipeline_logger=logger,
            message=f"Step completed: {step_name}",
        )

        return {
            "step_name": step_name,
            "status": EXECUTION_STATUS_SUCCESS,
            "message": "Step completed successfully.",
            "duration_seconds": duration_seconds,
            "critical": is_critical,
        }

    except Exception as error:

        finished_at = datetime.now()
        duration_seconds = (
            finished_at - started_at
        ).total_seconds()

        error_message = str(error)
        error_stacktrace = traceback.format_exc()

        write_pipeline_log(
            log_id=str(uuid.uuid4()),
            execution_id=JOB_RUN_ID,
            notebook_name=NOTEBOOK_NAME,
            layer_name=step_layer,
            entity_name=step_entity,
            target_table="not_applicable",
            status=EXECUTION_STATUS_FAILED,
            message=f"Step failed: {step_name} | {error_message}",
            started_at=started_at,
            finished_at=finished_at,
            duration_seconds=duration_seconds,
            records_read=None,
            records_written=None,
        )

        log_error(
            pipeline_logger=logger,
            message=f"Step failed: {step_name}",
            error=error,
        )

        return {
            "step_name": step_name,
            "status": EXECUTION_STATUS_FAILED,
            "message": error_message,
            "duration_seconds": duration_seconds,
            "critical": is_critical,
            "stacktrace": error_stacktrace,
        }

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Execute Incremental CEAP Steps

# COMMAND ----------

incremental_results = []

incremental_started_at = datetime.now()

for step_config in INCREMENTAL_STEPS:

    step_result = run_notebook_step(
        step_config=step_config,
    )

    incremental_results.append(step_result)

    if (
        step_result["status"] == EXECUTION_STATUS_FAILED
        and step_result["critical"]
        and FAIL_ON_STEP_ERROR
    ):

        raise Exception(
            f"Critical incremental CEAP step failed: "
            f"{step_result['step_name']} | "
            f"{step_result['message']}"
        )

incremental_finished_at = datetime.now()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Incremental CEAP Summary

# COMMAND ----------

total_steps = len(incremental_results)

success_steps = len([
    result
    for result in incremental_results
    if result["status"] == EXECUTION_STATUS_SUCCESS
])

warning_steps = len([
    result
    for result in incremental_results
    if result["status"] == EXECUTION_STATUS_WARNING
])

failed_steps = len([
    result
    for result in incremental_results
    if result["status"] == EXECUTION_STATUS_FAILED
])

incremental_duration_seconds = (
    incremental_finished_at - incremental_started_at
).total_seconds()

print("=" * 90)
print("INCREMENTAL CEAP PIPELINE SUMMARY")
print("=" * 90)
print(f"Job Run ID: {JOB_RUN_ID}")
print(f"Total steps: {total_steps}")
print(f"Successful steps: {success_steps}")
print(f"Warning steps: {warning_steps}")
print(f"Failed steps: {failed_steps}")
print(f"Pipeline duration seconds: {incremental_duration_seconds}")
print("=" * 90)

# COMMAND ----------

# ============================================================
# FINAL JOB LOG
# ============================================================

final_status = (
    EXECUTION_STATUS_FAILED
    if failed_steps > 0
    else EXECUTION_STATUS_SUCCESS
)

write_pipeline_log(
    log_id=str(uuid.uuid4()),
    execution_id=JOB_RUN_ID,
    notebook_name=NOTEBOOK_NAME,
    layer_name=LAYER_NAME,
    entity_name=ENTITY_NAME,
    target_table=PIPELINE_LOG_TABLE,
    status=final_status,
    message=(
        f"Incremental CEAP pipeline finished. "
        f"success={success_steps}, "
        f"warning={warning_steps}, "
        f"failed={failed_steps}"
    ),
    started_at=incremental_started_at,
    finished_at=incremental_finished_at,
    duration_seconds=incremental_duration_seconds,
    records_read=None,
    records_written=None,
)

# COMMAND ----------

if failed_steps > 0:

    print(
        f"WARNING: Incremental CEAP pipeline finished with "
        f"{failed_steps} failed step(s)."
    )

    if FAIL_ON_STEP_ERROR:
        raise Exception(
            f"Incremental CEAP pipeline failed with "
            f"{failed_steps} failed step(s)."
        )

print("INCREMENTAL CEAP PIPELINE EXECUTION COMPLETED")

BRAZIL LEGISLATIVE ANALYTICS MEDALLION
02 - RUN INCREMENTAL EXPENSES CEAP
Execution Timestamp: 2026-05-20 04:43:07.468782
Catalog: brazil_legislative_analytics
Environment: dev


2026-05-20 04:43:08 | INFO | JOBS | jobs.02_run_incremental_expenses_ceap | Starting step: validate_project_setup
2026-05-20 04:43:31 | INFO | JOBS | jobs.02_run_incremental_expenses_ceap | [SUCCESS] Step completed: validate_project_setup
2026-05-20 04:43:33 | WARNING | JOBS | jobs.02_run_incremental_expenses_ceap | Step skipped: validate_api_connection
2026-05-20 04:43:34 | WARNING | JOBS | jobs.02_run_incremental_expenses_ceap | Step skipped: bronze_despesas_ceap
2026-05-20 04:43:36 | WARNING | JOBS | jobs.02_run_incremental_expenses_ceap | Step skipped: silver_despesas_ceap
2026-05-20 04:43:37 | WARNING | JOBS | jobs.02_run_incremental_expenses_ceap | Step skipped: silver_fornecedores
2026-05-20 04:43:40 | WARNING | JOBS | jobs.02_run_incremental_expenses_ceap | Step skipped: gold_dm_fornecedor
2026-05-20 04:43:42 | WARNING | JOBS | jobs.02_run_incremental_expenses_ceap | Step skipped: gold_ft_despesas_ceap
2026-05-20 04:43:44 | WARNING | JOBS | jobs.02_run_incremental_expenses_ceap

INCREMENTAL CEAP PIPELINE SUMMARY
Job Run ID: 3504b5bc-f684-4110-b742-126c4ae3e2e2
Total steps: 12
Successful steps: 5
Warning steps: 7
Failed steps: 0
Pipeline duration seconds: 110.477459
INCREMENTAL CEAP PIPELINE EXECUTION COMPLETED


utils_logger loaded successfully.


PROJECT CONFIGURATION LOADED SUCCESSFULLY
PROJECT_NAME: brazil_legislative_analytics
PROJECT_VERSION: v1.0.0
PROJECT_ENVIRONMENT: dev
CATALOG_NAME: brazil_legislative_analytics
RUN_ID: e2ba1e99-38a2-4b97-bb59-48cc786c9c83


utils_config loaded successfully.


utils_config loaded successfully.


utils_table_logger loaded successfully.
